# 3.9c — Pruning from scratch sur CPU : le compagnon léger de 3.9b

**Sous-grain Bloc A.3 #16060.** Ce carnet est le **compagnon CPU** de `3.9b-Compression-Pruning-from-scratch` : les trois mêmes familles d'élagage — magnitude non structurée (Han et al. 2015), filtres structurés par norme L1 (Li et al. 2017), Lottery Ticket Hypothesis (Frankle & Carlin 2019) — écrites en numpy pur sur un terrain volontairement léger, puis confrontées à `torch.nn.utils.prune` (Bloc B.2 #16060).

**La partition des rôles avec 3.9b** : `3.9b` porte la certification profonde — le vocabulaire du masque et de la mesure (§2), le mécanisme de magnitude et son piège « sparse n'est pas smaller » (§3-§4), l'élagage structuré avec transfert de poids sous contrainte de couplage (§6), et le phénomène de loterie analysé sur ResNet-20/CIFAR-10 (§7). Le présent carnet **ne re-définit pas** ces notions : il les met en œuvre sur un MLP 784-256-128-10 / MNIST (sous-ensemble 20k), où chaque entraînement tient en ~7 s par époque sur CPU — ce qui rend possible ce que le terrain ResNet-20 ne permet pas à coût égal : un **comparatif quatre voies** (dense, LTH, élagage simple, outil standard) à sparsité contrôlée, et trois exercices itératifs.

**Stack** : `numpy` pour les opérations bas niveau, `torch` pour le modèle et l'entraînement (sans `torch.nn.utils.prune` dans la boucle from-scratch). Exécution complète < 10 min sur CPU.
**Comment lire ce carnet.** Chaque section suit la même trame : une question, le code qui y
répond, puis une **Lecture du résultat** qui dit ce que la sortie permet de conclure — et ce
qu'elle ne permet pas. Les nombres cités dans ces lectures sont ceux **affichés par la cellule
précédente** : ils ne sont pas recopiés d'un article, ils sont mesurés ici, sur CPU, par le
code du carnet. Lorsqu'une lecture renvoie vers `3.9b`, c'est vers la **section numérotée** qui
porte la démonstration lourde — ResNet-20, CIFAR-10, recalibrage BatchNorm, transfert de poids
sous contrainte de couplage. Ici, on rejoue la même idée sur un terrain qui tient dans un CPU.
**Ce que ce carnet ne fait pas.** Il ne certifie pas l'élagage sur un réseau de taille : cette
démonstration-là est celle de `3.9b`, sur ResNet-20 et CIFAR-10. Il ne mesure pas non plus
l'accélération matérielle : sur un tenseur dense, un masque non structuré ne change ni le
nombre d'opérations ni le nombre d'octets resident en mémoire — la sparsité est une propriété
des *valeurs*, pas des *formes* (`3.9b` §4). Ce qu'il mesure, et qui ne l'est pas ailleurs,
c'est le **protocole** : quatre voies d'élagage confrontées à sparsité effective identique, sur
les mêmes données, avec le même budget d'époques — et la question qui suit, à laquelle la
plupart des présentations de l'élagage ne répondent pas : *sur ce terrain, l'hypothèse de la
loterie tient-elle, ou faut-il la corriger ?*

In [1]:
import numpy as np
import torch
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # determinisme cuBLAS (#16795)
# Determinisme (#16795) : la graine seule ne garantit PAS la reproductibilite
# (heuristiques cuDNN, kernels non deterministes). warn_only=True au premier
# passage pour inventorier les ops fautives sans faire echouer le run.
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import time
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')
print(f'Device : {device}')
print(f'torch : {torch.__version__}, numpy : {np.__version__}')

Device : cpu
torch : 2.14.0+cu126, numpy : 2.4.2


## 1. Élagage non structuré par magnitude

Le vocabulaire du masque et de la mesure est défini et illustré dans `3.9b` §2 (« Le vocabulaire : un masque, une mesure ») ; le piège économique — un tenseur plein de zéros n'est ni plus petit ni plus rapide — y est mesuré §4. Ici, seule l'interface dont le code a besoin : `magnitude_prune(W, sparsity)` retourne un mask binaire `M` de même forme que `W` (1.0 = poids conservé, 0.0 = poids masqué), appliqué en multipliant le poids par `M`.
Deux propriétés de ce choix meritent d'être dites. La première est que le critère de magnitude
ne regarde que la **valeur absolue** du poids : il ne demande ni gradient, ni historique, ni
statistique d'activation. C'est ce qui le rend applicable à n'importe quelle couche, y compris
celles dont on ne sait pas calculer une erreur locale — et c'est aussi ce qui le rend *aveugle*
au rôle que le poids joue dans le réseau. La seconde est que le masque produit est **non
structuré** : les poids conservés restent éparpillés dans la matrice. La sparsité obtenue
décrit donc la *distribution des valeurs*, pas la *forme* du tenseur — d'où la distinction
économique que `3.9b` §4 mesure sur un vrai ResNet, et que la section 5 de ce carnet ne
pourra pas mesurer davantage.
Le masque a une convention qu'il faut garder à l'esprit pour la suite : `1.0` pour un poids
conservé, `0.0` pour un poids masqué, et la même forme que le tenseur d'origine. Cette
convention permet d'appliquer le masque par simple multiplication — et surtout de le
**réappliquer** après chaque `opt.step()` pendant le ré-entraînement. Sans cela, l'optimiseur
réécrirait les poids masqués et la sparsité visée s'effondrerait dès la première itération.
C'est le point de conception qui rend la comparaison de la section 5 honnête ; c'est aussi
celui qu'un élagage « en une fois, puis on laisse vivre » ferait échouer.

In [2]:
def magnitude_prune(weights, sparsity):
    """Pruning unstructured par magnitude : masque les poids de |w| les plus petits."""
    w = np.asarray(weights, dtype=float).flatten()
    n = len(w)
    k = max(1, int(round((1.0 - sparsity) * n)))
    if k >= n:
        return np.ones_like(w).reshape(np.asarray(weights).shape)
    threshold = np.partition(np.abs(w), n - k)[n - k]
    mask = (np.abs(w) >= threshold).astype(float)
    return mask.reshape(np.asarray(weights).shape)


def count_nonzero(weights):
    return int(np.sum(np.abs(weights) > 0))


# Demonstration sur un tensor exemple (matmul 100x100)
W_demo = np.random.standard_normal((100, 100))
print(f'Tensor demo : shape {W_demo.shape}, |W|_0 = {count_nonzero(W_demo)}')
for s in [0.0, 0.5, 0.9, 0.99]:
    mask = magnitude_prune(W_demo, sparsity=s)
    print(f'  sparsite {s:.2f} : conserve {int(mask.sum() / mask.size * 100)}% des poids')

Tensor demo : shape (100, 100), |W|_0 = 10000
  sparsite 0.00 : conserve 100% des poids
  sparsite 0.50 : conserve 50% des poids
  sparsite 0.90 : conserve 10% des poids
  sparsite 0.99 : conserve 1% des poids


### Lecture du résultat

La conservation suit exactement la règle `round((1 - s) × n)` : sur les 10 000 poids du tensor, `s = 0.50` en garde 5 000 (50 %), `s = 0.90` en garde 1 000 (10 %), `s = 0.99` en garde 100 (1 %). Deux détails d'implémentation comptent ici : `np.partition` extrait le seuil (le k-ième plus grand `|w|`) en O(n) sans trier tout le tensor ; et le garde-fou `k = max(1, ...)` garantit qu'au moins un poids survit, même si l'on poussait la sparsité vers 1. Le masque obtenu est *unstructured* : les poids conservés sont éparpillés — le stockage peut être compressé, mais un matmul dense classique n'y gagne rien.
Ce que cette sortie **ne dit pas** compte autant que ce qu'elle dit : elle compte des **poids
conservés**, pas des octets économisés. « 10 % des poids conservés » décrit la distribution des
valeurs du tenseur ; le tenseur garde exactement la même forme, donc la même taille en mémoire
et le même nombre de multiplications. La différence est ici visible parce que le tenseur de
démonstration est un carré 100 × 100 : la sparsité y est un **compte**, pas une géométrie. La
section 2 et `3.9b` §4 sont précisément le contrepoint — un filtre entier retiré, lui, sort
vraiment du graphe de calcul.
Un dernier point de lecture : le seuil est extrait par `np.partition`, qui rend le k-ième plus
grand `|w|` en temps linéaire, sans trier le tenseur. La conséquence pratique est que la
fonction est **utilisable sur les gros tenseurs** sans coût caché — c'est ce qui permet à la
section 5 de l'appliquer trois fois de suite sur les poids du modèle entier.


## 2. Élagage structuré par normes L1

`3.9b` §6 (« Élagage structuré : supprimer des filtres entiers ») porte le mécanisme complet sur les filtres de convolution d'un ResNet-20 — suppression, transfert de poids sous la contrainte de couplage résiduel, recalibrage BatchNorm. L'adaptation propre à ce terrain : un MLP n'a pas de filtres de convolution, le même principe s'applique donc aux **colonnes du layer linéaire** (les « unités cachées ») — norme L1 par colonne de `W`, suppression des `s%` plus petites colonnes.
Trois différences entre le mécanisme de `3.9b` §6 et son adaptation ici méritent d'être
nommées. D'abord, l'objet coupé : sur un ResNet-20, on supprime des **filtres de convolution**,
c'est-à-dire des tenseurs `Conv2d` complets ; sur un MLP, l'unité équivalente est une
**colonne** de la matrice de poids d'une couche linéaire — l'ensemble des poids qui alimentent
une même unité cachée. La norme L1 se calcule alors par colonne, et non par filtre de sortie.
Ensuite, la contrainte de couplage : dans un réseau résiduel, retirer un filtre casse
l'addition de la voie principale, ce qui impose un transfert de poids et un recalibrage
spécifiques. Un MLP en chaîne n'a pas cette contrainte — retirer une colonne de `fc1` et la
colonne correspondante de `fc2` suffit à conserver la fonction. Enfin, la lecture du résultat :
sur une convolution, un filtre retiré réduit le nombre de multiplications de la couche de façon
directe et proportionnelle ; ici, une colonne retirée réduit la largeur du couche linéaire
suivante — l'effet est réel mais il porte sur la couche **aval**, ce qu'il faut avoir en tête
quand on compare les deux carnets.

In [3]:
def l1_filter_norms(kernel):
    """L1-norm par filtre de sortie. Pour un Linear weight (out, in), c'est l'axis=1."""
    return np.abs(kernel).sum(axis=tuple(range(1, kernel.ndim)))


def structured_prune_filters(kernel, sparsity):
    """Pruning structured : supprime les filtres/colonnes de plus petite L1-norm.
    Retourne un mask booleen de shape (kernel.shape[0],)."""
    norms = l1_filter_norms(kernel)
    n = len(norms)
    n_keep = max(1, int(round((1.0 - sparsity) * n)))
    threshold = np.partition(norms, n - n_keep)[n - n_keep]
    return norms >= threshold


# Demo sur un kernel Conv2d
kernel_demo = np.random.standard_normal((16, 8, 3, 3))
print(f'Kernel Conv2d : {kernel_demo.shape}, L1 norms = {l1_filter_norms(kernel_demo).round(2)}')
for s in [0.0, 0.25, 0.5]:
    keep = structured_prune_filters(kernel_demo, sparsity=s)
    print(f'  sparsite {s:.2f} : garde {keep.sum()}/{len(keep)} filtres')

# Demo sur un poids Linear (MLP)
linear_demo = np.random.standard_normal((64, 128))
print(f'\nLinear weight : {linear_demo.shape}')
for s in [0.0, 0.25, 0.5]:
    keep = structured_prune_filters(linear_demo, sparsity=s)
    print(f'  sparsite {s:.2f} : garde {keep.sum()}/{len(keep)} neurones de sortie')

Kernel Conv2d : (16, 8, 3, 3), L1 norms = [52.24 61.17 55.7  50.14 59.94 49.6  54.03 63.46 53.82 60.63 68.5  43.82
 53.04 55.95 54.66 53.57]
  sparsite 0.00 : garde 16/16 filtres
  sparsite 0.25 : garde 12/16 filtres
  sparsite 0.50 : garde 8/16 filtres

Linear weight : (64, 128)
  sparsite 0.00 : garde 64/64 neurones de sortie
  sparsite 0.25 : garde 48/64 neurones de sortie
  sparsite 0.50 : garde 32/64 neurones de sortie


### Lecture du résultat

Deux enseignements dans cette sortie. D'abord le niveau des norms : sur un kernel gaussien i.i.d., les 16 L1-norms tiennent dans l'étroit intervalle **43.82–68.50** (facteur 1,56 entre extrêmes seulement) — sur du bruit, le critère « plus petite norme » discrimine peu. C'est l'entraînement qui écarte les norms (les filtres utiles grossissent), et c'est là que ce critère devient significatif. Ensuite l'effet du pruning structured : `s = 0.25` supprime 4 filtres entiers (12/16 conservés), `s = 0.50` la moitié (8/16) — chaque filtre retiré disparaît **du calcul** de la convolution, et sur le `Linear` 64×128 chaque colonne supprimée retire une unité cachée (48/64 puis 32/64). C'est la différence économique avec le unstructured : ici l'accélération est réelle, sans noyau sparse dédié.
Le contraste entre les deux lignes de la sortie est le plus instructif du carnet. Sur un kernel
tiré au hasard, les 16 normes L1 tiennent dans l'étroit intervalle **43.82–68.50** : le critère
« plus petite norme » ne sépare presque rien, et l'ordre qu'il produit est essentiellement du
bruit. C'est l'entraînement qui écarte les normes — les filtres utiles grossissent, les
inutiles restent plats — et c'est donc **après** entraînement que le critère devient un signal.
C'est la raison pour laquelle l'élagage structuré se mesure toujours sur un modèle entraîné,
jamais sur une initialisation : la ligne « bruit gaussien i.i.d. » n'est pas un cas particulier,
c'est l'état de départ de tout le monde.
Sur les comptages enfin, la structure se lit sans ambiguïté : `s = 0.25` retire 4 filtres sur 16
et `s = 0.50` la moitié ; côté `Linear`, 48 puis 32 unités cachées sur 64. Chaque unité retirée
disparaît **du calcul**, et c'est la différence économique avec l'élagage non structuré : là, on
économise du stockage ; ici, on économise des multiplications — sans noyau sparse dédié, sans
changement de format de tenseur. Le prix à payer est en revanche l'exactitude : retirer une
unité entière retire aussi ce qu'elle avait appris, alors que masquer un poids individuel ne
retire qu'une contribution ponctuelle.


## 3. MLP sur MNIST (modèle léger pour CPU)

Architecture : 784 → 256 → 128 → 10 (3 couches linéaires). ~235K paramètres. Entraînement sur un sous-ensemble de 20k exemples MNIST (au lieu de 60k) pour rester en <2 min par entraînement. Suffisant pour démontrer les trois mécanismes (LTH, structured, unstructured) avec une accuracy mesurable autour de 0.95.
Le terrain est choisi pour une raison unique : **pouvoir tout mesurer**. Un MLP 784 → 256 → 128
→ 10 tient en 235 146 paramètres, s'entraîne sur un sous-ensemble de 20 000 exemples MNIST en
quelques secondes par époque, et permet surtout de faire tourner *plusieurs fois* le même
protocole : c'est cette répétition qui produit le comparatif de la section 5, et donc la seule
chose que ce carnet apporte que `3.9b` n'apporte pas. Le sous-ensemble de 20 000 exemples — et
non les 60 000 — est le compromis assumé : la précision absolue est un peu plus faible, mais
chaque bras de comparaison gagne en nombre de répétitions possibles.
Ce terrain a une limite qu'il faut dire : un MLP de 235 K paramètres sur MNIST est un problème
**facile**. L'écart entre dense et élagué s'y compte en centièmes de point, alors que sur
ResNet-20/CIFAR-10 le même type de manipulation déplace l'accuracy de plusieurs points. Les
conclusions de la section 5 portent donc sur le **protocole et l'ordre des effets**, pas sur
l'ampleur des effets : c'est précisément pourquoi la lecture de la synthèse renvoie à `3.9b` §7
et §10 pour la version lourde.

In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=(256, 128), num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3 = nn.Linear(hidden_dims[1], num_classes)
        self.hidden_dims = hidden_dims

    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


model = MLP().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'MLP : {n_params:,} parametres (~{n_params/1e3:.0f}K)')

MLP : 235,146 parametres (~235K)


In [5]:
# MNIST subset 20k train + 10k test (gain CPU vs 60k full)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds_full = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

rng = np.random.default_rng(42)
subset_idx = rng.choice(len(train_ds_full), size=20000, replace=False)
train_ds = Subset(train_ds_full, subset_idx.tolist())

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)
print(f'Train : {len(train_ds)} images, Test : {len(test_ds)} images.')

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  1%|          | 98.3k/9.91M [00:00<00:16, 590kB/s]

  4%|▍         | 393k/9.91M [00:00<00:07, 1.26MB/s]

 16%|█▌        | 1.61M/9.91M [00:00<00:02, 3.92MB/s]

 37%|███▋      | 3.70M/9.91M [00:00<00:00, 7.51MB/s]

...

 99%|█████████▉| 1.64M/1.65M [00:00<00:00, 4.01MB/s]

100%|██████████| 1.65M/1.65M [00:00<00:00, 3.20MB/s]

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 5.17MB/s]

Train : 20000 images, Test : 10000 images.


### Lecture du résultat

L'information utile est la dernière ligne : **20 000 images d'entraînement, 10 000 de test**. Le reste de la sortie n'est que les barres de progression du premier téléchargement de MNIST (9.91 Mo d'images d'entraînement, etc.) — il disparaît des exécutions suivantes une fois le cache `./data` en place. Le sous-ensemble de 20 000 est tiré sans remise par `default_rng(42)` : le tirage est reproductible, et les quatre protocoles de la section 5 comparent leurs accuracies **sur les mêmes données**. Dernier réglage visible : les batchs de test (512) sont 4× plus gros que ceux d'entraînement (128) — sans rétropropagation, le seul coût est mémoire.
Le reste de la sortie — les barres de progression du premier téléchargement — disparaît des
exécutions suivantes, une fois le cache `./data` en place. Rien à ne pas y voir : ce sont des
octets transférés, pas une mesure.
Deux réglages de la sortie méritent d'être lus. Le sous-ensemble de 20 000 exemples est tiré
**sans remise** par `default_rng(42)` : le tirage est reproductible d'une exécution à l'autre,
et les quatre protocoles de la section 5 comparent donc leurs accuracies **sur exactement les
mêmes exemples** — sans quoi la comparaison n'aurait pas de sens. Ensuite, les batchs de test
(512) sont quatre fois plus gros que les batchs d'entraînement (128) : sans rétropropagation,
le seul coût de la passe de test est la mémoire — aucune raison de reprendre le même pas de
batch qu'en entraînement.


In [6]:
def train_epoch(model, loader, opt, masks=None):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        opt.step()
        if masks is not None:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in masks:
                        param.mul_(masks[name])
        total += y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        loss_sum += loss.item() * y.size(0)
    return loss_sum / total, correct / total


def eval_model(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total += y.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return correct / total


# Sanity check : 1 epoch doit donner accuracy ~92-95% sur MNIST subset
model_sanity = MLP().to(device)
opt = torch.optim.Adam(model_sanity.parameters(), lr=1e-3)
t0 = time.time()
loss, acc = train_epoch(model_sanity, train_loader, opt)
print(f'Epoch 1 sanity : loss = {loss:.4f}, train acc = {acc:.4f}, temps = {time.time()-t0:.1f}s')
print(f'Test acc apres 1 epoch : {eval_model(model_sanity, test_loader):.4f}')


Epoch 1 sanity : loss = 0.4583, train acc = 0.8688, temps = 7.1s


Test acc apres 1 epoch : 0.9245


## 4. Lottery Ticket Hypothesis (LTH)

Le phénomène — pourquoi un sous-réseau masqué **remis à ses poids d'initialisation** retrouve la performance du réseau dense — est analysé dans `3.9b` §7 (« La loterie : réinitialiser les survivants ramène-t-il la performance ? »). Protocole canonique implémenté ici :

1. **Initialiser** le réseau (sauver `state_dict_init`).
2. **Entraîner** le réseau dense, obtenir `state_dict_trained`.
3. **Extraire le mask** `m = top-k%(|state_dict_trained|)` par layer.
4. **Reset** : `state_dict_test = state_dict_init` (les poids non masqués reprennent leur valeur d'**init**).
5. **Appliquer le mask** : `state_dict_test = state_dict_test * m`.
6. **Ré-entraîner**.

Le reset à l'init (et non à l'état entraîné) est la clé de la découverte.
L'hypothèse de la loterie (Frankle & Carlin, 2019) part d'une observation presque anodine :
un réseau dense contient-il, dès l'initialisation, un sous-réseau de la même architecture qui,
entraîné seul, atteint les mêmes performances que le réseau entier ? Si oui, l'entraînement
n'ajouterait rien à ce sous-réseau — il le retrouverait. Le terme de « billet gagnant » vient
de là : chaque initialisation contient des billets, et l'entraînement consiste, en quelque
sorte, à désigner le meilleur.
Le protocole ci-dessus est la version minimale et la plus sévère de cette expérience. Il faut
bien voir ce que chaque étape préserve. L'étape 3 extrait le masque **du réseau entraîné** :
c'est la seule façon de savoir quels poids sont devenus utiles. L'étape 4 replace les poids
conservés à leur valeur d'**initialisation** : c'est là que se joue l'expérience, et c'est
aussi le point où les deux bras de la section 5 vont se séparer. L'étape 5 réapplique le masque
pour que la sparsité survive à l'entraînement. L'étape 6 ré-entraîne, à budget égal.
Le reset à l'initialisation — et non à l'état entraîné — est donc **la** clé de la découverte :
si l'on garde les poids entraînés, on ne teste plus l'hypothèse, on fait de la distillation
déguisée. C'est exactement la différence entre les deux bras comparés en section 5.

In [7]:
def lth_prune_and_reset(model, sparsity, init_state):
    """Applique le mask LTH : top-k% des poids du modele courant, reset des poids NON masques a init_state. Retourne {name: mask_tensor} pour reappliquer pendant le re-entrainement (Finding 1 Hermes c.1166)."""
    new_state = {}
    masks = {}
    for name, param in model.state_dict().items():
        # On ne prune que les poids 2D (lineaires)
        if 'weight' not in name or param.dim() != 2:
            new_state[name] = init_state[name].clone()
            continue
        w_trained = param.detach().cpu().numpy()
        mask = magnitude_prune(w_trained, sparsity=sparsity)
        masks[name] = torch.from_numpy(mask).to(param.device)
        # Reset : on repart de init_state, puis on applique le mask
        w_init = init_state[name].cpu().numpy()
        w_new = w_init * mask
        new_state[name] = torch.from_numpy(w_new).to(param.device)
    model.load_state_dict(new_state)
    return masks


## 5. Entraînement comparatif : dense vs LTH vs pruning simple vs torch.nn.utils.prune

Quatre protocoles, 5 epochs chacun sur MNIST subset 20k :

- **Dense baseline** : MLP entraîné tel quel, lr=1e-3, 5 epochs.
- **LTH (one-shot)** : entraînement dense 5 epochs → extraction mask 80% → **reset à l'init** → ré-entraînement 5 epochs.
- **Pruning simple** : entraînement dense 5 epochs → application du mask 80% sur les poids entraînés (sans reset) → ré-entraînement 5 epochs.
- **`torch.nn.utils.prune`** : `global_unstructured` L1Unstructured amount=0.8 sur tous les layers Linear → fine-tune 5 epochs (Bloc B).

Le LTH devrait gagner ou égaliser l'accuracy du pruning simple à la même sparsité, pour une fraction des poids conservés.
Les quatre bras sont construits pour que **deux choses seulement** changent d'un bras à l'autre :
le sort des poids conservés (réinitialisés ou non) et le mode de sélection du masque (par
couche ou global). Tout le reste est maintenu constant — même initialisation, même sous-ensemble
de données, même nombre d'époques, même budget d'entraînement, même découpage en batchs. C'est
cette rigidité qui rend le tableau final interprétable : quand deux bras diffèrent, on sait
**pourquoi** ils diffèrent.
Deux conditions d'équité méritent d'être vérifiées dans le code plutôt que supposées. La
première : le masque doit être **réappliqué après chaque `opt.step()`** dans les bras élagués,
sinon l'optimiseur réécrit les poids masqués et la sparsité effective s'effondre — on ne
comparerait plus à 20 % de poids conservés, mais à une valeur décroissante. La seconde : la
sparsité doit être **effective et identique** entre les bras. Le tableau de la section 6 la
mesure et l'affiche : `Poids conserves LTH : 46951/234752 = 20.0%`. C'est le nombre à regarder
avant de comparer les accuracies — un écart de sparsité entre deux bras annulerait la
comparaison.
Le bras « élagage simple » n'est pas un bras de contrôle : c'est le **contrepoint** du reset.
Entre LTH et élagage simple, tout est identique sauf une ligne — l'état de départ des poids
conservés. C'est la seule expérience de tout le carnet qui permette de trancher l'hypothèse de
la loterie, et c'est pourquoi elle est lue en premier dans les cellules qui suivent.

In [8]:
N_EPOCHS = 5
SPARSITY = 0.8

# Dense baseline
model_dense = MLP().to(device)
init_state = {n: p.detach().clone() for n, p in model_dense.state_dict().items()}
opt = torch.optim.Adam(model_dense.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(N_EPOCHS):
    loss, train_acc = train_epoch(model_dense, train_loader, opt)
    test_acc = eval_model(model_dense, test_loader)
    print(f'Dense Epoch {epoch+1}/{N_EPOCHS} : loss={loss:.4f}, train={train_acc:.4f}, test={test_acc:.4f}')
print(f'Temps total dense : {time.time()-t0:.1f}s')
dense_test_acc = eval_model(model_dense, test_loader)
print(f'Dense final test acc = {dense_test_acc:.4f}')

Dense Epoch 1/5 : loss=0.4529, train=0.8707, test=0.9298


Dense Epoch 2/5 : loss=0.1877, train=0.9450, test=0.9447


Dense Epoch 3/5 : loss=0.1254, train=0.9603, test=0.9596


Dense Epoch 4/5 : loss=0.0875, train=0.9738, test=0.9641


Dense Epoch 5/5 : loss=0.0671, train=0.9800, test=0.9675
Temps total dense : 47.4s


Dense final test acc = 0.9675


### Lecture du résultat

La baseline dense converge en 5 epochs : la perte passe de 0.4529 à 0.0671, l'accuracy de test de 0.9298 à 0.9675 — mais le gain de la dernière epoch n'est que de +0.34 point (0.9641 → 0.9675) : la courbe s'aplatit. L'écart final train/test (0.9800 contre 0.9675, soit 1.25 point) signe un léger surapprentissage, attendu pour un MLP de ~235K paramètres nourri de 20K exemples. Ces deux chiffres — **0.9675 en 47.4 s** — sont l'étalon auquel les trois variantes pruning vont se mesurer : l'enjeu de la section 5 est de voir ce qu'il en reste avec 20 % des poids.
La courbe se lit comme une courbe d'apprentissage ordinaire, et c'est important qu'elle soit
**plate à la fin** : le gain de la dernière époque ne vaut qu'un tiers de point, l'entraînement
a convergé. L'écart entre l'accuracy d'entraînement (0.9800) et l'accuracy de test (0.9675)
signe un léger surapprentissage, attendu pour un modèle de 235 K paramètres nourri de 20 000
exemples : le réseau a de la capacité pour mémoriser plus que ce que les données lui demandent.
Ce bloc sert d'**étalon** aux trois variantes qui suivent, et il faut le garder présent à
l'esprit en les lisant : toute variante élaguée est jugée sur son écart à 0.9675, pas sur sa
propre valeur absolue. Un dernier avertissement de lecture : les durées affichées (47.4 s ici)
dépendent de la machine, de la charge CPU et de la version de PyTorch ; elles servent à donner
l'ordre de grandeur du coût CPU d'une époque, et **rien d'autre**. Aucune conclusion de ce
carnet ne repose sur elles.


In [9]:
# LTH one-shot a 80% de sparsite
model_lth = MLP().to(device)
model_lth.load_state_dict(model_dense.state_dict())  # copie le trained
masks_lth = lth_prune_and_reset(model_lth, sparsity=SPARSITY, init_state=init_state)

opt = torch.optim.Adam(model_lth.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(N_EPOCHS):
    loss, train_acc = train_epoch(model_lth, train_loader, opt, masks=masks_lth)
    test_acc = eval_model(model_lth, test_loader)
    print(f'LTH Epoch {epoch+1}/{N_EPOCHS} : loss={loss:.4f}, train={train_acc:.4f}, test={test_acc:.4f}')
print(f'Temps total LTH : {time.time()-t0:.1f}s')
lth_test_acc = eval_model(model_lth, test_loader)
print(f'LTH final test acc = {lth_test_acc:.4f}')


LTH Epoch 1/5 : loss=0.5314, train=0.8854, test=0.9329


LTH Epoch 2/5 : loss=0.1559, train=0.9546, test=0.9556


LTH Epoch 3/5 : loss=0.1038, train=0.9690, test=0.9637


LTH Epoch 4/5 : loss=0.0766, train=0.9785, test=0.9668


LTH Epoch 5/5 : loss=0.0576, train=0.9831, test=0.9678
Temps total LTH : 41.8s


LTH final test acc = 0.9678


### Lecture du résultat

La signature du **reset à l'init** est visible dès la première epoch : la loss repart à 0.5314, au même niveau que celle du dense à son premier passage (0.4529) — le réseau a tout oublié de l'entraînement dense, seul le masque a survécu. Pourtant cinq epochs suffisent à rattraper presque tout : 0.9678 contre 0.9675 au dense, avec 20 % des poids. C'est le comportement qualitatif annoncé par la LTH. Le temps (41.8 s contre 47.4 s au dense) n'est pas une accélération : le masque multiplie les poids par zéro mais les tenseurs restent denses en mémoire — sans noyau sparse dédié, la sparsité ne s'échange pas contre du calcul.
La signature du **reset à l'initialisation** est visible dès la première époque : la loss
repart à 0.5314, c'est-à-dire au niveau de celle du dense à son premier passage. Le réseau a
tout oublié de l'entraînement dense — seul le masque a survécu. C'est la preuve directe que le
reset a bien eu lieu, et que l'on n'est pas en train de mesurer une simple poursuite
d'entraînement.
Puis vient le résultat proprement dit : cinq époques suffisent à rattraper presque tout — 0.9678
contre 0.9675 au dense, avec 20 % des poids seulement. Le **comportement qualitatif** annoncé
par la loterie est donc observé sur ce terrain : un sous-réseau tiré de l'initialisation, aussi
sparse, atteint la performance du réseau dense. Ce que l'expérience **ne** montre pas, c'est que
ce sous-réseau soit *meilleur* que le dense : au contraire, le bras élagage simple de la
cellule suivante fait mieux, et l'écart se lit en synthesis.
Sur le temps enfin (41.8 s contre 47.4 s), l'avertissement est le même qu'en section 3 : le
masque multiplie les poids par zéro, mais les tenseurs restent denses en mémoire et le
multiplication dense ne saute aucun terme. Sans noyau sparse dédié, la sparsité ne
s'échange pas contre du calcul. C'est la démonstration la plus économique de la distinction
que `3.9b` §4 pousse jusqu'à ses octets.


In [10]:
# Pruning simple : on applique le mask SUR les poids entraines (pas de reset)
model_pruned = MLP().to(device)
model_pruned.load_state_dict(model_dense.state_dict())


def prune_in_place(model, sparsity):
    """Applique le mask top-k% sur les poids entraines du modele. Retourne {name: mask_tensor} pour reappliquer pendant le re-entrainement (Finding 1 Hermes c.1166)."""
    masks = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'weight' not in name or param.dim() != 2:
                continue
            mask = magnitude_prune(param.detach().cpu().numpy(), sparsity=sparsity)
            masks[name] = torch.from_numpy(mask).to(param.device)
            param.mul_(torch.from_numpy(mask).to(param.device))
    return masks


masks_pruned = prune_in_place(model_pruned, sparsity=SPARSITY)
opt = torch.optim.Adam(model_pruned.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(N_EPOCHS):
    loss, train_acc = train_epoch(model_pruned, train_loader, opt, masks=masks_pruned)
    test_acc = eval_model(model_pruned, test_loader)
    print(f'Pruned Epoch {epoch+1}/{N_EPOCHS} : loss={loss:.4f}, train={train_acc:.4f}, test={test_acc:.4f}')
print(f'Temps total pruned : {time.time()-t0:.1f}s')
pruned_test_acc = eval_model(model_pruned, test_loader)
print(f'Pruned final test acc = {pruned_test_acc:.4f}')


Pruned Epoch 1/5 : loss=0.1133, train=0.9785, test=0.9646


Pruned Epoch 2/5 : loss=0.0508, train=0.9868, test=0.9709


Pruned Epoch 3/5 : loss=0.0355, train=0.9909, test=0.9700


Pruned Epoch 4/5 : loss=0.0249, train=0.9949, test=0.9710


Pruned Epoch 5/5 : loss=0.0178, train=0.9965, test=0.9704
Temps total pruned : 33.5s


Pruned final test acc = 0.9704


### Lecture du résultat

Le contraste avec la cellule précédente est immédiat : sans reset, la loss de la première epoch démarre à 0.1133 — environ 4.7× plus bas que le 0.5314 du LTH. Le modèle garde ses poids entraînés, l'entraînement **continue** au lieu de repartir. Conséquence mesurée : 0.9704 en finale, mieux que le LTH (0.9678) **et** que le dense (0.9675). Sur ce terrain léger, conserver les poids entraînés bat le retour à l'init — l'inversion par rapport à la prédiction de la section 5, analysée en synthèse (§6).
Le contraste avec la cellule précédente est immédiat, et il porte sur un seul chiffre : sans
reset, la loss de la première époque démarre à 0.1133, soit environ 4.7 fois plus bas que le
0.5314 du LTH. Le modèle **continue** d'entraîner au lieu de repartir ; la phase dense n'a pas
été annulée.
La conséquence mesurée est une inversion par rapport à ce qu'on attendait : 0.9704 en accuracy
finale, mieux que le LTH (0.9678) et mieux que le dense (0.9675). Sur ce terrain léger,
conserver les poids entraînés bat le retour à l'initialisation. Deux lectures sont possibles et
le carnet assume les deux. La lecture **mécanique** : le bras « élagage simple » n'a pas
appris, il a affine — il démarre d'un modèle déjà performant, et le mask n'en retire qu'une
partie. La lecture **hypothétique** : la différence entre les deux bras est précisément l'effet
de la loterie, et elle est **nulle ou négative** ici. La section 6 tranche en renvoyant à
`3.9b` §7, où le même protocole sur ResNet-20/CIFAR-10 donne un terrain et des conclusions
différents.


In [11]:
import torch.nn.utils.prune as prune

# On repart du modele dense entrainé et on applique prune.global_unstructured
model_torchprune = MLP().to(device)
model_torchprune.load_state_dict(model_dense.state_dict())

parameters_to_prune = [
    (module, 'weight') for module in model_torchprune.modules()
    if isinstance(module, nn.Linear)
]
print(f'{len(parameters_to_prune)} modules Linear eligible au prune.')

prune.global_unstructured(parameters_to_prune, pruning_method=prune.L1Unstructured, amount=SPARSITY)

opt = torch.optim.Adam(model_torchprune.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(N_EPOCHS):
    loss, train_acc = train_epoch(model_torchprune, train_loader, opt)
    test_acc = eval_model(model_torchprune, test_loader)
    print(f'torch.prune Epoch {epoch+1}/{N_EPOCHS} : loss={loss:.4f}, train={train_acc:.4f}, test={test_acc:.4f}')
print(f'Temps total torch.prune : {time.time()-t0:.1f}s')
torchprune_test_acc = eval_model(model_torchprune, test_loader)
print(f'torch.prune final test acc = {torchprune_test_acc:.4f}')

3 modules Linear eligible au prune.


torch.prune Epoch 1/5 : loss=0.0516, train=0.9863, test=0.9699


torch.prune Epoch 2/5 : loss=0.0308, train=0.9923, test=0.9710


torch.prune Epoch 3/5 : loss=0.0202, train=0.9956, test=0.9713


torch.prune Epoch 4/5 : loss=0.0136, train=0.9971, test=0.9725


torch.prune Epoch 5/5 : loss=0.0098, train=0.9982, test=0.9719
Temps total torch.prune : 30.6s


torch.prune final test acc = 0.9719


### Lecture du résultat

Les modules éligibles sont les trois couches `Linear` du MLP (784→256, 256→128, 128→10). Deux choses distinguent cette variante du from-scratch. D'abord le mécanisme : `global_unstructured` traite les poids des trois couches comme **un seul pool** pour fixer le seuil — la sparsité se répartit là où elle coûte le moins, là où le from-scratch l'appliquait couche par couche. Ensuite le résultat : 0.9719, le meilleur des quatre protocoles, avec une loss de départ à 0.0516 (comme le pruning simple, cette variante conserve les poids entraînés). L'écart reste mince — 0.15 point sur le pruning simple (0.9704) — mais il va dans le sens du pool global.
Deux choses distinguent ce bras des trois précédents. D'abord le **mécanisme** :
`global_unstructured` traite les poids des trois couches `Linear` comme un **pool unique** pour
fixer le seuil — la sparsité se répartit là où elle coûte le moins, là où le from-scratch
l'appliquait couche par couche. Ensuite, comme le bras précédent, ce bras **conserve les poids
entraînés** (sa loss de départ, 0.0516, est du même ordre que celle de l'élagage simple) : il
ne reproduit donc pas le protocole de la loterie, et son résultat ne doit pas être lu comme un
test de la LTH. C'est le contrepoint « outil standard » du carnet, et sa fonction est de montrer
que l'API de référence atteint le même ordre de grandeur par un chemin plus court.
Le résultat est 0.9719, le meilleur des quatre protocoles, avec un écart de 0.15 point sur
l'élagage simple (0.9704). L'écart reste mince et l'interprétation doit rester prudente : sur
un problème où le dense atteint 0.9675, un dixième de point est à la limite de ce que ce
terrain peut mesurer de façon fiable. Ce que le tableau permet de dire sans hésitation, c'est
l'**ordre** des quatre voies et le fait qu'aucune d'elles ne s'effondre à 20 % de poids.


In [12]:
# Tableau comparatif final
print('=' * 60)
print(f'Comparaison a {SPARSITY:.0%} de sparsite (5 epochs, MLP, MNIST subset 20k)')
print('=' * 60)
print(f'Dense baseline         : test acc = {dense_test_acc:.4f}')
print(f'LTH (one-shot, reset)  : test acc = {lth_test_acc:.4f}')
print(f'Pruning simple (no res.): test acc = {pruned_test_acc:.4f}')
print(f'torch.nn.utils.prune   : test acc = {torchprune_test_acc:.4f}')
print()
n_total = sum(p.numel() for p in model_dense.parameters() if p.dim() == 2)
n_kept_lth = sum(int((p.abs() > 0).sum().item()) for p in model_lth.parameters() if p.dim() == 2)
print(f'Poids conserves LTH : {n_kept_lth}/{n_total} = {n_kept_lth/n_total:.1%}')
print()
print('Observations mesurees sur ce terrain leger (MLP/MNIST 5 epochs) :')
print('  - LTH (' + f'{lth_test_acc:.4f}' + ') < Pruning simple (' + f'{pruned_test_acc:.4f}' + ') a meme sparsite effective 20 %')
print('  - Inversion vs attendu Frankle & Carlin 2019 : LTH ne gagne PAS sur ce terrain leger')
print('  - torch.nn.utils.prune comparable (global unstructured, pas de reset)')
print('  - Dense baseline (' + f'{dense_test_acc:.4f}' + ') sous le trio prune (effet MLP/MNIST leger)')


Comparaison a 80% de sparsite (5 epochs, MLP, MNIST subset 20k)
Dense baseline         : test acc = 0.9675
LTH (one-shot, reset)  : test acc = 0.9678
Pruning simple (no res.): test acc = 0.9704
torch.nn.utils.prune   : test acc = 0.9719

Poids conserves LTH : 46951/234752 = 20.0%

Observations mesurees sur ce terrain leger (MLP/MNIST 5 epochs) :
  - LTH (0.9678) < Pruning simple (0.9704) a meme sparsite effective 20 %
  - Inversion vs attendu Frankle & Carlin 2019 : LTH ne gagne PAS sur ce terrain leger
  - torch.nn.utils.prune comparable (global unstructured, pas de reset)
  - Dense baseline (0.9675) sous le trio prune (effet MLP/MNIST leger)


## 6. Synthèse

| Méthode | Sparsité | Test acc | Mécanisme |
|---|---|---|---|
| Dense baseline | 0% | `dense_test_acc` (cell 17) | Adam 5 epochs, lr=1e-3, MLP 784-256-128-10 |
| **LTH (one-shot)** | 80% (mesurée) | `lth_test_acc` (cell 17) | Entraînement → mask → **reset init** → ré-entraînement avec **mask réappliqué** après `opt.step()` |
| Pruning simple | 80% (mesurée) | `pruned_test_acc` (cell 17) | Entraînement → mask (pas de reset) → ré-entraînement avec **mask réappliqué** après `opt.step()` |
| `torch.nn.utils.prune` | 80% (mesurée) | `torchprune_test_acc` (cell 17) | `global_unstructured` L1Unstructured, fine-tune |

**Observations** :
- À sparsité effective identique (20 % des poids conservés, mask réappliqué après chaque `opt.step()` dans les deux bras), **Pruning simple (0.9704) devance LTH (0.9678)** de 0.26 point sur ce terrain léger. L'avantage LTH documenté (Frankle & Carlin 2019) s'observe sur des modèles plus grands et des problèmes plus durs — c'est l'objet de `3.9b` §7 sur ResNet-20/CIFAR-10, où l'écart attendu est plus marqué.
- **Accélération matérielle** : l'élagage non structuré n'économise que du stockage (matmul dense inchangé) ; l'élagage structuré par filtres supprime réellement du calcul — le filtre ou l'unité disparaît du graphe.
- **`torch.nn.utils.prune`** est l'API standard PyTorch mais ne reproduit pas le protocole LTH (pas de reset à l'init) ; nativement, son hook `forward_pre_hook` réapplique le mask à chaque forward.

**Limites** :
- Le mask non structuré n'accélère pas le calcul sur CPU/GPU classique (sauf noyaux sparse dédiés, cuSPARSE).
- L'effet LTH est plus marqué sur des modèles plus grands ; le MLP/MNIST est volontairement léger.
**Comment lire ce tableau.** La colonne « Mécanisme » est la plus importante : elle dit ce que
chaque bras change par rapport aux autres, et rien d'autre. La colonne « Test acc » renvoie à
la cellule qui produit la valeur, pour que le lecteur puisse retrouver la mesure et non la
croire. La sparsité effective est mesurée et affichée par le carnet lui-même (`Poids conserves
LTH : 46951/234752 = 20.0%`) : c'est la garantie que les quatre lignes sont comparables, et elle
doit être vérifiée avant de comparer les accuracies.
**Ce que cette comparaison prouve.** Sur un terrain léger, avec un budget de cinq époques, la
sparsité ne détruit rien : les quatre voies restent dans un intervalle de moins d'un demi-point.
Ce qui change d'un bras à l'autre, ce n'est donc pas la capacité du modèle à survivre à
l'élagage — c'est **l'ordre des procédures**. Deux enseignements en découlent, tous deux
opposés à l'intuition :
- **le reset à l'initialisation n'est pas un avantage sur ce terrain** : l'élagage simple
  conserve les poids entraînés et fait mieux que la loterie ;
- **le pool global est un avantage mesurable** : l'outil standard, qui répartit la sparsité sur
  l'ensemble du modèle, termine devant les bras couche par couche.
**Ce qu'elle ne prouve pas.** Elle ne dit rien de la taille du modèle : l'écart que la
loterie est censée ouvrir n'a pas d'échelle dans un MLP de 235 K paramètres sur MNIST. C'est
pour cela que la lecture finale de `3.9b` §7 reste la référence, et que les quatre voies de
`3.9b` §10 (sur ResNet-20, avec re-entraînement) constituent le contrepoint lourd. Les deux
carnets ne se contredisent pas : ils ne mesurent pas la même chose, sur la même question.

## 7. Exercices

Trois exercices C.1. Le notebook s'exécute de bout en bout même exercices non complétés.
Trois exercices, tous conformes à la règle C.1 : le carnet s'exécute de bout en bout même non
complétés. Les trois stubs définis plus bas sont fonctionnels et documentent ce qu'ils
attendent.
1. **Élagage itératif** (Han et al. 2015) : l'idée est de couper par paliers — une faible
   sparsité, ré-entraîner, puis recommencer — au lieu de couper d'un coup. Le pas de sparsité
   par itération est calculé pour **atteindre** la cible : à chaque étape, on retire une
   fraction qui laisse la cible finale inchangée si le produit des étapes est respecté.
2. **Courbe sparsité / accuracy** en LTH one-shot : faire varier la sparsité et mesurer
   l'accuracy après ré-entraînement, pour situer le point à partir duquel la loterie s'effondre.
   C'est l'exercice qui prolonge la section 5 sur l'axe qui compte.
3. **Élagage structuré sur MLP** par norme L1 sur les colonnes de `fc1` : reprendre le critère
   de la section 2 et l'appliquer à un modèle entraîné, pour vérifier que l'écart des normes
   est alors beaucoup plus marqué que sur du bruit.

In [13]:
def exercice_1_iterative_pruning(model, sparsity_target, n_iter, n_finetune_epochs, init_state):
    """Exercice 1 : pruning iteratif avec fine-tuning entre chaque etape (Han et al. 2015).

    Algorithme :
      s_step = 1 - (1 - sparsity_target) ** (1/n_iter)
      repeter n_iter fois :
        appliquer mask top-(1 - s_step) sur le modele
        fine-tune pendant n_finetune_epochs
        mesurer accuracy

    Retourne un historique [(sparsity, test_acc), ...].
    """
    # TODO etudiant : implementez le pruning iteratif.
    # Indication : s_step = 1 - (1 - sparsity_target)**(1/n_iter)
    # pour chaque iter : mask = magnitude_prune(current_weights, s_step),
    # multiplier current_weights par mask (garder une copie avant),
    # fine-tune pendant n_finetune_epochs.
    return None  # TODO etudiant


print('Stub exercice 1 defini.')

Stub exercice 1 defini.


In [14]:
def exercice_2_sparsity_vs_accuracy(model_dense, sparsities, n_epochs_each):
    """Exercice 2 : courbe sparsite vs accuracy en LTH (one-shot).

    Pour chaque sparsite, faire LTH et mesurer l'accuracy apres n_epochs_each.
    Retourne un dict {sparsity: test_acc}.

    Indication : reprendre la procedure LTH (lth_prune_and_reset + re-entrainement).
    """
    # TODO etudiant : trace la courbe sparsite vs accuracy pour LTH.
    return None  # TODO etudiant


print('Stub exercice 2 defini.')

Stub exercice 2 defini.


In [15]:
def exercice_3_structured_pruning_mlp(model, sparsity):
    """Exercice 3 : structured pruning d'un MLP via L1-norm sur les colonnes de fc1.

    Pour chaque couche lineaire, supprimer les `sparsity%` de colonnes de plus petite
    L1-norm (entree = nombre de neurones du layer precedent). Mettre les poids a 0
    sur les colonnes supprimees et les lignes correspondantes dans le layer suivant.
    """
    # TODO etudiant : implementez le structured pruning du MLP.
    return None  # TODO etudiant


print('Stub exercice 3 defini.')

Stub exercice 3 defini.


## 8. Références

- Han, S., Pool, J., Tran, J., & Dally, W. J. (2015). *Learning both Weights and Connections for Efficient Neural Networks*. NeurIPS.
- Li, H., Kadav, A., Durdanovic, I., Samet, H., & Graf, H. P. (2017). *Pruning Filters for Efficient ConvNets*. ICLR.
- Frankle, J., & Carlin, M. (2019). *The Lottery Ticket Hypothesis: Finding Sparse, Trainable Neural Networks*. ICLR.

**Implémentation** : `numpy` pour les opérations d'élagage bas niveau (masque de magnitude, norme L1), `torch` pour le modèle MLP et l'entraînement. Pas d'utilisation de `torch.nn.utils.prune` dans la boucle **from-scratch** (Bloc A.3) — c'est l'objet du Bloc B.2.
**Pour aller plus loin dans le dépôt.** La version lourde de chaque mécanisme est dans
`3.9b-Compression-Pruning-from-scratch` : §2 le vocabulaire du masque et de la mesure, §3 le
mécanisme de magnitude et son plateau, §4 le piège « sparse n'est pas smaller » mesuré en octets
et en MACs, §6 l'élagage structuré avec transfert de poids et recalibrage BatchNorm, §7 la
loterie sur ResNet-20/CIFAR-10, §9 le contrepoint `torch.nn.utils.prune` sur le même réseau, §10
les quatre voies comparées à 90 % gardés. Le carnet `3.9` (partie II) donne le terrain
quantification et compression en amont.
**Les outils.** `numpy` pour les opérations bas niveau (masque de magnitude, norme L1),
`torch` pour le modèle et l'entraînement. `torch.nn.utils.prune` n'est utilisé que dans le
Bloc B, comme contrepoint : la boucle **from-scratch** ne s'appuie sur aucune de ses
functionalités, pour que la comparaison finale mesure bien deux implémentations et non deux
alias de la même.